# Fine-tuning

In [ ]:
!pip install -U torchao peft transformers

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from collections import defaultdict

In [ ]:
categories_to_ids = {
    'anatomical location': 0,
    'animal': 1,
    'biomedical technique': 2,
    'bacteria': 3,
    'chemical': 4,
    'dietary supplement': 5,
    'DDF': 6,
    'drug': 7,
    'food': 8,
    'gene': 9,
    'human': 10,
    'microbiome': 11,
    'statistical technique': 12,
    'none': 13
}

ids_to_categories = {v: k for k, v in categories_to_ids.items()}

In [ ]:
class_counts = {
    0: 0.048,
    1: 0.053,
    2: 0.053,
    3: 0.075,
    4: 0.152,
    5: 0.037,
    6: 0.337,
    7: 0.018,
    8: 0.013,
    9: 0.015,
    10: 0.087,
    11: 0.094,
    12: 0.018,
    13: 0.6
}

# Calculate weights: Higher weight for lower percentage
weights = []
for i in range(14):
    p = class_counts.get(i, 1.0) # default to 1.0 if missing
    w = 1.0 / (p / 100.0 + 1e-6)
    weights.append(w)

# Normalize weights so the average is 1.0
weight_tensor = torch.tensor(weights, dtype=torch.float)
weight_tensor = weight_tensor / weight_tensor.mean()

weight_tensor[13] = 0.5  # Forces the model to care about 'None' mistakes

In [ ]:
print(weight_tensor)

In [ ]:
class GBDataset(Dataset):
    def __init__(self, articles_df, annotations_df=None):
        self.annotations_df = annotations_df
        self.articles_df = articles_df

        self.dataset = []

        mismatches = 0

        for idx in range(self.articles_df.shape[0]):
            start_labels = torch.zeros(len(self.articles_df["input_ids"][idx]), dtype=torch.uint8)
            end_labels = torch.zeros(len(self.articles_df["input_ids"][idx]), dtype=torch.uint8)
            span_idxs = []
            span_labels = []
            start_token_to_orig = {}
            end_token_to_orig = {}
            pmid = self.articles_df["pmid"][idx]
            if self.annotations_df is not None:
                for row in self.annotations_df[self.annotations_df['pmid'] == pmid].itertuples(name=None):
                    start_token = None
                    end_token = None

                    ann_start_idx = row[2]
                    ann_end_idx = row[3]

                    # if the entity is in the abstract, shift the idx with the number of chars
                    # added at the start of the string for the title
                    if row[4] == 0:
                        ann_start_idx += self.articles_df['added_chars_no'][idx]
                        ann_end_idx += self.articles_df['added_chars_no'][idx]

                    for i, (offset_start, offset_end) in enumerate(self.articles_df["offset_mapping"][idx]):
                        # Skip special tokens like [CLS] or [SEP] which have (0,0) offsets
                        if offset_start == 0 and offset_end == 0:
                            continue


                        # 1. Capture the start: First token that contains char_start
                        if start_token is None and offset_start <= ann_start_idx < offset_end:
                            start_token = i

                        # 2. Capture the end: Token that contains the last character
                        if offset_start <= ann_end_idx < offset_end:
                            end_token = i


                    if start_token is not None and end_token is not None:
                        start_labels[start_token] = 1
                        end_labels[end_token] = 1
                        span_idxs.append([start_token, end_token])
                        span_labels.append(row[6])
                        start_token_to_orig[start_token] = ann_start_idx if row[4] == 1 else ann_start_idx - self.articles_df['added_chars_no'][idx]
                        end_token_to_orig[end_token] = ann_end_idx if row[4] == 1 else ann_end_idx - self.articles_df['added_chars_no'][idx]

                        # 1. Get the original text for comparison
                        original_entity = self.articles_df['text'][idx][ann_start_idx:ann_end_idx + 1]

                        # 2. Get the tokens from the input_ids
                        # input_ids is the tensor/list from the tokenizer encoding
                        token_ids = self.articles_df['input_ids'][idx][start_token : end_token + 1]
                        decoded_entity = tokenizer.decode(token_ids).strip()


                        # 3. Print and compare
                        if original_entity.lower() != decoded_entity.lower():
                            print(f"Mismatch found!")
                            print(f"Original: '{original_entity}'")
                            print(f"Decoded:  '{decoded_entity}'")
                            print(f"Indices:  {start_token}:{end_token}")
                            mismatches += 1

            self.dataset.append({
                "pmid":            pmid,
                "location":        self.annotations_df[self.annotations_df['pmid'] == pmid]['location'] if self.annotations_df is not None else None,
                "input_ids":       torch.tensor(self.articles_df.iloc[idx]['input_ids'], dtype=torch.long),
                "start_labels":    start_labels,
                "end_labels":      end_labels,
                "span_idxs":       span_idxs,
                "span_labels":     span_labels,
                "start_token_to_orig": start_token_to_orig,
                "end_token_to_orig": end_token_to_orig,
                "abstract_start_idx": self.articles_df['added_chars_no'][idx]
            })

    def __len__(self):
        return self.articles_df.shape[0]

    def __getitem__(self, idx):
        return self.dataset[idx]


In [ ]:
def collate_fn(batch):
    """
    Pad sequences and assemble per-batch tensors.

    match_labels is stored as a sparse list of (batch_idx, i, j) gold pairs
    """
    max_len = max(b["input_ids"].shape[0] for b in batch)
    B = len(batch)

    input_ids      = torch.zeros(B, max_len, dtype=torch.long)
    attention_mask = torch.zeros(B, max_len, dtype=torch.long)
    start_labels   = torch.zeros(B, max_len)
    end_labels     = torch.zeros(B, max_len)

    # Sparse gold span pairs for the matching loss.
    # Each entry is (batch_idx, start_token, end_token).
    gold_pairs: List[Tuple[int, int, int]] = []

    for i, b in enumerate(batch):
        L = b["input_ids"].shape[0]
        input_ids[i, :L]      = b["input_ids"]
        attention_mask[i, :L] = 1
        start_labels[i, :L]   = b["start_labels"]
        end_labels[i, :L]     = b["end_labels"]
        for (s, e) in b["span_idxs"]:
            gold_pairs.append((i, s, e))

    span_idxs   = [b["span_idxs"]   for b in batch]
    span_labels = [b["span_labels"]  for b in batch]

    return {
        "pmid":            [b["pmid"]     for b in batch],
        "location":        [b["location"] for b in batch],
        "input_ids":       input_ids,
        "attention_mask":  attention_mask,
        "start_labels":    start_labels,
        "end_labels":      end_labels,
        "gold_pairs":      gold_pairs,   # list of (b, s, e)
        "span_idxs":       span_idxs,
        "span_labels":     span_labels,
    }

## Model arhitecture

In [ ]:
@dataclass
class NERConfig:
    model_name: str = "thomas-sounack/BioClinical-ModernBERT-base"
    num_labels: int = 14          # 13 = non-entity, 0–12 = entity types
    max_seq_len: int = 512
    max_span_len: int = 20        # max token span to consider as candidate
    top_k_spans: int = 100        # top-k start-end pairs during inference
    bilinear_proj_dim: int = 128

    # LoRA
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = field(
        default_factory=lambda: ["Wqkv", "out_proj", "fc1", "fc2"]
    )

    # Training
    batch_size: int = 8
    grad_accum_steps: int = 4     # effective batch = 32
    lr_stage1: float = 2e-4
    lr_stage2: float = 1e-3
    lr_stage3: float = 5e-5
    warmup_ratio: float = 0.1
    epochs_stage1: int = 5
    epochs_stage2: int = 5
    epochs_stage3: int = 10

    # ── Boundary loss (Tan et al. AAAI 2020 §3.3) ──────────────────────────
    # L_boundary = lambda_s * L_start + lambda_e * L_end + lambda_m * L_match
    #
    # L_start / L_end: per-token binary cross-entropy
    #   label_i = 1 iff token i is the start (or end) of any gold entity span
    #
    # L_match: binary cross-entropy on the [L×L] bilinear score matrix
    #   match_label[i, j] = 1 iff (i, j) is an exact gold span (same entity)
    #   This directly trains the bilinear scorer to associate valid start-end pairs.
    #
    lambda_s: float = 1.0               # weight for start BCE loss
    lambda_e: float = 1.0               # weight for end BCE loss
    lambda_m: float = 0.5               # weight for bilinear matching BCE loss
    boundary_pos_weight: float = 10.0   # pos_weight for start/end BCE (sparsity)
    match_pos_weight: float = 20.0      # pos_weight for match BCE (even sparser)

    # Hard negative mining for span classification head
    # neg_ratio: how many hard negatives to sample per gold span in the batch.
    # neg_ratio=1 → balanced 50/50
    neg_ratio: int = 1

    span_label_smoothing: float = 0.1

    # Hardware
    fp16: bool = True
    device: str = "cuda"

CFG = NERConfig()

In [ ]:
def mine_hard_negatives(
    span_scores:  torch.Tensor,              # [B, L, L]  boundary bilinear scores
    attention_mask: torch.Tensor,            # [B, L]
    gold_span_idxs: List[List[Tuple[int,int]]],  # gold spans per example
    neg_ratio:    int,
    max_span_len: int,
) -> List[List[Tuple[int, int]]]:
    """
    For each example in the batch, selects up to neg_ratio * n_gold non-gold
    candidate spans with the highest boundary bilinear scores.

    These are the "hard negatives": spans the boundary head is most confident
    about but which are not real entities. Training the span head on them
    teaches it to output label 0 (None) for boundary false positives.

    Returns a list (one per example) of (start, end) hard-negative span indices.
    """
    B = span_scores.shape[0]
    scores_cpu = span_scores.detach().cpu()
    mask_cpu   = attention_mask.cpu()
    hard_negs: List[List[Tuple[int, int]]] = []

    for b in range(B):
        sl = int(mask_cpu[b].sum().item())
        gold_set = {tuple(s) for s in gold_span_idxs[b]}
        n_gold = len(gold_set)
        n_neg  = max(1, n_gold * neg_ratio)   # at least 1 negative even if no gold

        sc = scores_cpu[b, :sl, :sl]   # [sl, sl]

        # Mask to valid candidates: upper-triangular, span ≤ max_span_len
        cand_mask = torch.zeros(sl, sl, dtype=torch.bool)
        for i in range(sl):
            for j in range(i, min(sl, i + max_span_len + 1)):
                cand_mask[i, j] = True

        # Zero out gold positions so they cannot be selected as negatives
        for (gs, ge) in gold_set:
            if gs < sl and ge < sl:
                cand_mask[gs, ge] = False

        if not cand_mask.any():
            hard_negs.append([])
            continue

        # Score only the valid non-gold positions
        flat_scores = sc[cand_mask]               # [N_valid]
        cand_positions = cand_mask.nonzero(as_tuple=False)  # [N_valid, 2]

        # Take top-n_neg by boundary score (hardest negatives)
        k = min(n_neg, flat_scores.shape[0])
        top_k_idx = flat_scores.topk(k).indices   # indices into flat_scores

        negs = [(int(cand_positions[idx, 0]), int(cand_positions[idx, 1]))
                for idx in top_k_idx.tolist()]
        hard_negs.append(negs)

    return hard_negs


def build_span_inputs_with_negatives(
    gold_span_idxs:  List[List[Tuple[int, int]]],
    gold_span_labels: List[List[int]],
    hard_neg_idxs:   List[List[Tuple[int, int]]],
) -> Tuple[List[List[Tuple[int, int]]], List[List[int]]]:
    """
    Merges gold spans (labels 0-12) with hard negative spans (label 13)
    into a single input list for the span classification head.

    Returns:
        combined_idxs   — per-example list of (s, e)
        combined_labels — per-example list of int labels (13 for negatives)
    """
    combined_idxs:   List[List[Tuple[int, int]]] = []
    combined_labels: List[List[int]] = []

    for b_idx in range(len(gold_span_idxs)):
        idxs   = list(gold_span_idxs[b_idx])
        labels = list(gold_span_labels[b_idx])

        for (s, e) in hard_neg_idxs[b_idx]:
            idxs.append((s, e))
            labels.append(13)

        combined_idxs.append(idxs)
        combined_labels.append(labels)

    return combined_idxs, combined_labels

In [ ]:
class BoundaryHead(nn.Module):
    """
    Three supervised components (Tan et al. AAAI 2020 §3.3):

      1. start_clf  — binary classifier: is token i the start of some entity?
      2. end_clf    — binary classifier: is token j the end of some entity?
      3. bilinear   — learned matching scorer: does (i, j) form a valid span?
    """

    def __init__(self, hidden_size: int, proj_dim: int):
        super().__init__()
        self.start_clf = nn.Linear(hidden_size, 1)
        self.end_clf   = nn.Linear(hidden_size, 1)
        # Factored bilinear projections: score(i,j) ≈ (W_s h_i)·(W_e h_j)
        self.W_s = nn.Linear(hidden_size, proj_dim, bias=False)
        self.W_e = nn.Linear(hidden_size, proj_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(
        self,
        hidden: torch.Tensor,   # [B, L, H]
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns:
            start_logits  [B, L]
            end_logits    [B, L]
            span_scores   [B, L, L]
        """
        start_logits = self.start_clf(hidden).squeeze(-1)   # [B, L]
        end_logits   = self.end_clf(hidden).squeeze(-1)     # [B, L]

        s_proj = self.W_s(hidden)   # [B, L, d]
        e_proj = self.W_e(hidden)   # [B, L, d]

        span_scores = torch.bmm(s_proj, e_proj.transpose(-1, -2)) + self.bias

        return start_logits, end_logits, span_scores



In [ ]:
class SpanClassificationHead(nn.Module):
    """
    Mean-pools token representations for a span [s, e] and classifies it.
    Works on a variable list of (start, end) candidate spans per example.
    """

    def __init__(self, hidden_size: int, num_labels: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels),
        )

    def pool_span(
        self,
        hidden: torch.Tensor,    # [L, H]
        start: int,
        end: int,
    ) -> torch.Tensor:           # [H]
        return hidden[start : end + 1].mean(0)

    def forward(
        self,
        hidden: torch.Tensor,                    # [B, L, H]
        span_idxs: List[List[Tuple[int, int]]],  # per-example list of (s, e)
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            logits   [N_total_spans, num_labels]
            batch_id [N_total_spans]   — which example each span belongs to
        """
        all_logits, batch_ids = [], []
        for b_idx, spans in enumerate(span_idxs):
            if not spans:
                continue
            pooled = torch.stack(
                [self.pool_span(hidden[b_idx], s, e) for s, e in spans]
            )                          # [N_b, H]
            all_logits.append(self.mlp(pooled))
            batch_ids.extend([b_idx] * len(spans))

        if not all_logits:
            dummy = torch.zeros(0, self.mlp[-1].out_features, device=hidden.device)
            return dummy, []

        return torch.cat(all_logits, 0), batch_ids


In [ ]:
class DualHeadNER(nn.Module):
    """
    Full model: LoRA-wrapped encoder + boundary head + span classification head.
    """

    def __init__(self, cfg: NERConfig):
        super().__init__()
        self.cfg = cfg

        # Load base encoder and apply LoRA
        base_encoder = AutoModel.from_pretrained(cfg.model_name, torch_dtype=torch.float16)
        lora_cfg = LoraConfig(
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=cfg.lora_dropout,
            bias="none",
            target_modules=cfg.lora_target_modules,
        )
        self.encoder = get_peft_model(base_encoder, lora_cfg)
        self.encoder.gradient_checkpointing_enable()

        H = self.encoder.config.hidden_size
        self.boundary_head = BoundaryHead(H, cfg.bilinear_proj_dim)
        self.span_head     = SpanClassificationHead(H, cfg.num_labels)

    def encode(self, input_ids, attention_mask):
        return self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state      # [B, L, H]

    def forward(
        self,
        input_ids:       torch.Tensor,
        attention_mask:  torch.Tensor,
        span_idxs:       Optional[List] = None,   # None → skip span head
    ):
        hidden = self.encode(input_ids, attention_mask)
        start_logits, end_logits, span_scores = self.boundary_head(hidden)

        span_logits, batch_ids = None, None
        if span_idxs is not None:
            span_logits, batch_ids = self.span_head(hidden, span_idxs)

        return {
            "hidden":       hidden,
            "start_logits": start_logits,
            "end_logits":   end_logits,
            "span_scores":  span_scores,
            "span_logits":  span_logits,
            "batch_ids":    batch_ids,
        }

In [ ]:
def boundary_loss(
    start_logits:   torch.Tensor,              # [B, L]
    end_logits:     torch.Tensor,              # [B, L]
    span_scores:    torch.Tensor,              # [B, L, L]
    start_labels:   torch.Tensor,             # [B, L]
    end_labels:     torch.Tensor,             # [B, L]
    gold_pairs:     List[Tuple[int, int, int]], # (batch_idx, s, e) — sparse
    attention_mask: torch.Tensor,             # [B, L]
    cfg: NERConfig,
) -> torch.Tensor:
    """
    Three-component boundary loss (Tan et al. AAAI 2020 §3.3):

        L_boundary = λ_s · L_start + λ_e · L_end + λ_m · L_match

    L_start, L_end
        Per-token BCE restricted to non-padding positions.

    L_match — we evaluate BCE only at the ~B·L·max_span_len valid candidate
        positions (upper-triangular, non-padding, span ≤ max_span_len)
    """
    device = start_logits.device
    seq_mask = attention_mask.bool()          # [B, L]
    B, L = start_logits.shape

    # ── L_start and L_end ────────────────────────────────────────────────────
    pw_se = torch.tensor(cfg.boundary_pos_weight, device=device)
    bce_se = nn.BCEWithLogitsLoss(pos_weight=pw_se, reduction="none")
    loss_s = bce_se(start_logits, start_labels)[seq_mask].mean()
    loss_e = bce_se(end_logits,   end_labels)  [seq_mask].mean()

    # ── L_match — sparse candidate-only BCE ──────────────────────────────────
    # Build (batch_idx, i, j) index tensors for all valid candidate pairs.
    # A valid pair satisfies: i <= j, j-i <= max_span_len, both non-padding.
    seq_lens = attention_mask.sum(dim=1).tolist()   # actual length per example

    cand_b, cand_i, cand_j = [], [], []
    for b in range(B):
        sl = int(seq_lens[b])
        for i in range(sl):
            j_max = min(sl, i + cfg.max_span_len + 1)
            for j in range(i, j_max):
                cand_b.append(b)
                cand_i.append(i)
                cand_j.append(j)

    if not cand_b:
        # Degenerate batch — return only start/end losses
        return cfg.lambda_s * loss_s + cfg.lambda_e * loss_e

    cand_b = torch.tensor(cand_b, dtype=torch.long, device=device)
    cand_i = torch.tensor(cand_i, dtype=torch.long, device=device)
    cand_j = torch.tensor(cand_j, dtype=torch.long, device=device)

    # Gather scores at candidate positions: [N_cands]
    cand_scores = span_scores[cand_b, cand_i, cand_j]

    # Build match labels: 0 for all candidates, 1 for gold pairs
    cand_labels = torch.zeros(len(cand_b), device=device)

    if gold_pairs:
        # Index gold pairs into the flat candidate list.
        # Build a lookup: (b, i, j) -> flat index for O(1) marking.
        pair_to_idx = {
            (int(b), int(i), int(j)): k
            for k, (b, i, j) in enumerate(
                zip(cand_b.tolist(), cand_i.tolist(), cand_j.tolist())
            )
        }
        for (gb, gs, ge) in gold_pairs:
            key = (int(gb), int(gs), int(ge))
            if key in pair_to_idx:
                cand_labels[pair_to_idx[key]] = 1.0

    pw_m = torch.tensor(cfg.match_pos_weight, device=device)
    loss_m = F.binary_cross_entropy_with_logits(
        cand_scores, cand_labels,
        pos_weight=pw_m,
        reduction="mean",
    )

    return cfg.lambda_s * loss_s + cfg.lambda_e * loss_e + cfg.lambda_m * loss_m

In [ ]:
def span_loss(
    span_logits:     torch.Tensor,
    span_labels:     List[List[int]],
    label_smoothing: float,
    weight_tensor: torch.Tensor
) -> torch.Tensor:
    """
    CE loss over gold spans (0-12) and hard negative spans (13).
    label_smoothing only applied to non-zero labels to avoid
    pulling negatives away from a clean 13 target.
    """
    flat_labels = torch.tensor(
        [lbl for lbls in span_labels for lbl in lbls],
        dtype=torch.long, device=span_logits.device,
    )
    if flat_labels.numel() == 0:
        return span_logits.sum() * 0.0

    # Move weights to the correct device
    weights = weight_tensor.to(span_logits.device)


    return F.cross_entropy(span_logits, flat_labels, weight=weights, label_smoothing=label_smoothing)

## Training loop

In [ ]:
def freeze(module: nn.Module):
    for p in module.parameters():
        p.requires_grad_(False)

def unfreeze(module: nn.Module):
    for p in module.parameters():
        p.requires_grad_(True)

def set_stage(model: DualHeadNER, stage: int):
    """
    Stage 1: boundary head + LoRA active; span head frozen
    Stage 2: span head + LoRA active; boundary head frozen
    Stage 3: everything active
    """
    if stage == 1:
        unfreeze(model.encoder)      # LoRA params only (PEFT handles base freeze)
        unfreeze(model.boundary_head)
        freeze(model.span_head)

    elif stage == 2:
        freeze(model.boundary_head)
        freeze(model.encoder)
        unfreeze(model.span_head)

    elif stage == 3:
        unfreeze(model.encoder)
        unfreeze(model.boundary_head)
        unfreeze(model.span_head)

    # LoRA base weights stay frozen in all stages
    for name, param in model.encoder.named_parameters():
        if "lora_" not in name:
            param.requires_grad_(False)

In [ ]:
@torch.no_grad()
def predict(
    model:     DualHeadNER,
    batch:     dict,
    cfg:       NERConfig,
    threshold: float = 0.8,
) -> List[List[Tuple[int, int, int]]]:
    """
    Returns per-example list of (start, end, label) predictions.

    Pipeline:
      1. Boundary head → start_probs, end_probs, bilinear span_scores
      2. Combined score = start_prob(i) * end_prob(j) * sigmoid(span_score(i,j))
      3. Top-k candidates above threshold → span head → entity labels
    """
    device = next(model.parameters()).device
    ids  = batch["input_ids"].to(device)
    mask = batch["attention_mask"].to(device)

    use_fp16 = cfg.fp16 and device.type == "cuda"
    with torch.amp.autocast("cuda", enabled=use_fp16):
        out = model(ids, mask, span_idxs=None) # boundary only
    start_probs = torch.sigmoid(out["start_logits"])  # [B, L]
    end_probs   = torch.sigmoid(out["end_logits"])    # [B, L]
    span_scores = out["span_scores"]                  # [B, L, L]
    hidden      = out["hidden"]

    B, L = ids.shape
    results = []

    for b in range(B):
        seq_len = mask[b].sum().item()
        s_p = start_probs[b, :seq_len]
        e_p = end_probs[b, :seq_len]
        sc  = span_scores[b, :seq_len, :seq_len]  # [seq_len, seq_len]

        # Valid candidate mask: i <= j, span length <= max_span_len
        cand_mask = torch.tril(
            torch.ones(seq_len, seq_len, device=device, dtype=torch.bool),
            diagonal=cfg.max_span_len,
        ) & torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool))

        combined = s_p.unsqueeze(1) * e_p.unsqueeze(0) * torch.sigmoid(sc)
        combined = combined * cand_mask.float()

        # Top-k
        flat = combined.view(-1)
        k = min(cfg.top_k_spans, (flat > 0).sum().item())
        if k == 0:
            results.append([])
            continue

        top_vals, top_idx = flat.topk(k)
        starts = (top_idx // seq_len).tolist()
        ends   = (top_idx %  seq_len).tolist()
        cands  = [(s, e) for s, e, v in zip(starts, ends, top_vals.tolist())
                  if v > threshold]

        if not cands:
            results.append([])
            continue

        # Classify candidates with span head
        span_logits, _ = model.span_head(hidden[b:b+1], [cands])
        pred_labels = span_logits.argmax(-1).tolist()

        preds = []
        for (s, e), lbl in zip(cands, pred_labels):
            if lbl != 13:   # 13 = non-entity
                preds.append((s, e, lbl))
        results.append(preds)

    return results

In [ ]:
@torch.no_grad()
def evaluate(
    model:  DualHeadNER,
    loader: DataLoader,
    cfg:    NERConfig,
) -> dict:
    model.eval()
    device = next(model.parameters()).device

    # Track stats per class: {label_id: {"tp": 0, "fp": 0, "fn": 0}}
    class_stats = {i: {"tp": 0, "fp": 0, "fn": 0} for i in range(0, 13)}

    for batch in loader:
        preds_batch = predict(model, batch, cfg)

        for b_idx, preds in enumerate(preds_batch):
            gold = set(
                (s, e, lbl)
                for (s, e), lbl in zip(
                    batch["span_idxs"][b_idx],
                    batch["span_labels"][b_idx],
                )
                if lbl != 13
            )
            pred_set = set(preds)

            # True Positives
            for span in (pred_set & gold):
                class_stats[span[2]]["tp"] += 1

            # False Positives (predicted but not in gold)
            for span in (pred_set - gold):
                class_stats[span[2]]["fp"] += 1

            # False Negatives (gold but not predicted)
            for span in (gold - pred_set):
                class_stats[span[2]]["fn"] += 1

    # --- Metrics Calculation ---
    total_tp = total_fp = total_fn = 0
    macro_prec = macro_rec = 0

    for lbl, stats in class_stats.items():
        tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
        total_tp += tp
        total_fp += fp
        total_fn += fn

        # Per-class Precision/Recall for Macro
        p = tp / (tp + fp + 1e-9)
        r = tp / (tp + fn + 1e-9)
        macro_prec += p
        macro_rec += r

    # Micro-Average
    micro_prec = total_tp / (total_tp + total_fp + 1e-9)
    micro_rec  = total_tp / (total_tp + total_fn + 1e-9)
    micro_f1   = 2 * micro_prec * micro_rec / (micro_prec + micro_rec + 1e-9)

    # Macro-Average
    macro_prec /= 13
    macro_rec /= 13
    macro_f1 = 2 * macro_prec * macro_rec / (macro_prec + macro_rec + 1e-9)

    return {
        "micro_f1": micro_f1,
        "micro_precision": micro_prec,
        "micro_recall": micro_rec,
        "macro_f1": macro_f1,
        "class_stats": class_stats
    }


In [ ]:
def save_checkpoint(state, stage, epoch, path="checkpoint.pth"):
    # Save to a temporary file
    torch.save(state, f"stage_{stage}_epoch_{epoch}_{path}")
    # Maintain a 'latest' symlink or copy for easy resumption
    torch.save(state, f"latest_stage_{stage}.pth")
    print(f"  --> Checkpoint saved: stage_{stage}_epoch_{epoch}")

In [ ]:
def make_optimizer(model: DualHeadNER, lr: float):
    trainable = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-2)


def train_stage(
    model:      DualHeadNER,
    loader:     DataLoader,
    stage:      int,
    epochs:     int,
    lr:         float,
    cfg:        NERConfig,
    val_loader: Optional[DataLoader] = None,
    scaler:     Optional[torch.cuda.amp.GradScaler] = None,
):
    set_stage(model, stage)
    optimizer = make_optimizer(model, lr)
    total_steps  = (len(loader) // cfg.grad_accum_steps) * epochs
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    device   = torch.device(cfg.device)
    use_fp16 = cfg.fp16 and device.type == "cuda"
    model.to(device)
    if use_fp16 and scaler is None:
        scaler = torch.cuda.amp.GradScaler()

    print(f"\n{'='*55}")
    print(f"Stage {stage} | {epochs} epochs | lr={lr}")
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_trainable:,}")
    print('='*55)

    model.train()
    optimizer.zero_grad()

    for epoch in range(epochs):
        epoch_loss = 0.0

        for step, batch in tqdm(enumerate(loader)):
            ids     = batch["input_ids"].to(device)
            mask    = batch["attention_mask"].to(device)
            s_lbl   = batch["start_labels"].to(device)
            e_lbl   = batch["end_labels"].to(device)
            g_pairs = batch["gold_pairs"]        # list of (b,s,e) tuples

            span_idxs   = batch["span_idxs"]   if stage >= 2 else None
            span_labels = batch["span_labels"]  if stage >= 2 else None

            with torch.amp.autocast("cuda", enabled=use_fp16):
                out = model(ids, mask, span_idxs=span_idxs)

                loss = torch.tensor(0.0, device=device)

                if stage in (1, 3):
                    loss = loss + boundary_loss(
                        out["start_logits"], out["end_logits"], out["span_scores"],
                        s_lbl, e_lbl, g_pairs, mask,
                        cfg,
                    )

                if stage in (2, 3) and out["span_logits"] is not None:
                    # ── Hard negative mining ──────────────────────────────
                    # Mine top-scoring non-gold spans from boundary scores.
                    hard_negs = mine_hard_negatives(
                        out["span_scores"],
                        mask,
                        batch["span_idxs"],
                        cfg.neg_ratio,
                        cfg.max_span_len,
                    )

                    # Combine gold spans + hard negatives for the span head
                    combined_idxs, combined_labels = build_span_inputs_with_negatives(
                        batch["span_idxs"],
                        batch["span_labels"],
                        hard_negs,
                    )

                    # Run span head on combined inputs
                    # (re-use hidden from the forward pass above)
                    span_logits, _ = model.span_head(out["hidden"], combined_idxs)

                    if span_logits.shape[0] > 0:
                        loss = loss + span_loss(
                            span_logits, combined_labels, cfg.span_label_smoothing, weight_tensor
                        )

                loss = loss / cfg.grad_accum_steps

            if use_fp16:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % cfg.grad_accum_steps == 0:
                if use_fp16:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                optimizer.zero_grad()
                scheduler.step()


            epoch_loss += loss.item() * cfg.grad_accum_steps

        avg = epoch_loss / len(loader)
        print(f"  Epoch {epoch+1}/{epochs} | loss={avg:.4f}")

        if val_loader is not None:
            val = evaluate(model, val_loader, cfg)
            print(f"  Val{val}")
            model.train()

        checkpoint_state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'stage': stage,
        }
        if use_fp16:
            checkpoint_state['scaler_state_dict'] = scaler.state_dict()

        save_checkpoint(checkpoint_state, stage, epoch)

    return scaler

In [ ]:
def train_pipeline(
    train_bronze_ds: GBDataset,
    train_silver_gold_ds: GBDataset,
    val_ds:   GBDataset,
    cfg:      NERConfig = CFG,
) -> DualHeadNER:
    """
    End-to-end: initialise model, run stages 1–3, return trained model.
    """
    model = DualHeadNER(cfg)

    train_bronze_loader = DataLoader(
        train_bronze_ds, batch_size=cfg.batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )
    train_silver_gold_loader = DataLoader(
        train_silver_gold_ds, batch_size=cfg.batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )

    scaler = torch.amp.GradScaler('cuda') if cfg.fp16 else None

    # Stage 1 — boundary head
    print('Starting stage 1...')
    scaler = train_stage(
        model, train_bronze_loader, stage=1,
        epochs=cfg.epochs_stage1, lr=cfg.lr_stage1,
        cfg=cfg, val_loader=val_loader, scaler=scaler,
    )
    print('Finished stage 1')

    # Stage 2 — span classification head (boundary frozen)
    print('Starting stage 2...')
    scaler = train_stage(
        model, train_bronze_loader, stage=2,
        epochs=cfg.epochs_stage2, lr=cfg.lr_stage2,
        cfg=cfg, val_loader=val_loader, scaler=scaler,
    )
    print('Finished stage 2')

    # Stage 3 — joint fine-tuning
    print('Starting stage 3...')
    scaler = train_stage(
        model, train_silver_gold_loader, stage=3,
        epochs=cfg.epochs_stage3, lr=cfg.lr_stage3,
        cfg=cfg, val_loader=val_loader, scaler=scaler,
    )
    print('Finished stage 3')

    return model

## Predictions

In [ ]:
def format_predictions(
    model:     DualHeadNER,
    loader:    DataLoader,
    tokenizer: AutoTokenizer,
    cfg:       NERConfig,
) -> pd.DataFrame:
    model.eval()
    results = {}

    for batch in loader:
        preds_batch = predict(model, batch, cfg)

        for b_idx, preds in enumerate(preds_batch):
            pmid     = batch["pmid"][b_idx]
            ids      = batch["input_ids"][b_idx].tolist()
            start_token_to_orig = batch["start_token_to_orig"][b_idx]
            end_token_to_orig = batch["end_token_to_orig"][b_idx]
            abstract_start_idx = batch["abstract_start_idx"][b_idx]
            results[pmid] = {
                "entities": []
            }

            for (s, e, lbl) in preds:
                span_ids = ids[s : e + 1]
                text     = tokenizer.decode(span_ids, skip_special_tokens=True)
                text = text.removesuffix(" ,.;:-()[]")
                text = text.removeprefix(" ,.;:-()[]")

                results[pmid]["entities"].append({
                    "start_idx": start_token_to_orig[s],
                    "end_idx":   end_token_to_orig[e],
                    "location": "abstract" if start_token_to_orig[s] >= abstract_start_idx else "title",
                    "text_span":   text,
                    "label":       lbl
                })

    return results

## Run

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
train_bronze_ds = torch.load("./datasets/bronze_dataset.pt", weights_only=False)
train_silver_ds = torch.load("./datasets/silver_dataset.pt", weights_only=False)
train_silver_2025_ds = torch.load("./datasets/silver2025_dataset.pt", weights_only=False)
train_gold_ds = torch.load("./datasets/gold_dataset.pt", weights_only=False)
train_silver_gold_ds = ConcatDataset([train_silver_ds, train_silver_2025_ds, train_gold_ds])
val_ds = torch.load("./datasets/dev_dataset.pt", weights_only=False)

In [ ]:
test_ds     = torch.load("./datasets/test_dataset.pt", weights_only=False)
test_loader = DataLoader(test_ds, batch_size=16, collate_fn=collate_fn)

In [ ]:
model = train_pipeline(train_bronze_ds, train_silver_gold_ds, val_ds, CFG)
torch.save(model.state_dict(), "gutbrainie_model.pt")

In [ ]:
def _label_name(lbl: int) -> str:
    return ids_to_categories.get(lbl, f"Label-{lbl}")


@torch.no_grad()
def error_analysis(
    model:     DualHeadNER,
    loader:    DataLoader,
    tokenizer: AutoTokenizer,
    cfg:       NERConfig,
    max_examples: int = 500,
) -> pd.DataFrame:
    """
    Runs the model over `loader` and returns a DataFrame of every error,
    with one row per wrong span. Columns:

        pmid, location, span_text,
        start_token, end_token,
        error_type   : "FP" (false positive) | "FN" (false negative)
                       | "WRONG_LABEL" (span found but label wrong)
        pred_label, pred_label_name,
        gold_label, gold_label_name,
        boundary_score   : combined boundary score for this span
        span_head_conf   : softmax probability of the predicted label

    Understanding the error types:
        FP          — span predicted but not in gold; almost always a boundary
                      head false positive that the span head failed to reject.
                      After hard-negative training, FP counts should drop
                      because the span head now outputs 13 (None) for these.
        FN          — gold span not predicted; either the boundary head missed
                      it (low combined score) or the span head classified it
                      as None (label 13).
        WRONG_LABEL — span boundaries correct but category wrong; tells you
                      which label pairs confuse the model most.
    """
    model.eval()
    device = next(model.parameters()).device
    rows = []
    n_examples = 0

    for batch in loader:
        if n_examples >= max_examples:
            break

        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        # Forward: boundary + all-candidate span scores for diagnostics
        out = model(ids, mask, span_idxs=None)
        start_probs = torch.sigmoid(out["start_logits"])
        end_probs   = torch.sigmoid(out["end_logits"])
        span_scores = out["span_scores"]
        hidden      = out["hidden"]

        B = ids.shape[0]

        for b in range(B):
            if n_examples >= max_examples:
                break

            pmid     = batch["pmid"][b]
            location = batch["location"][b]
            token_ids = batch["input_ids"][b].tolist()

            sl  = int(mask[b].sum().item())
            s_p = start_probs[b, :sl]
            e_p = end_probs[b, :sl]
            sc  = span_scores[b, :sl, :sl]

            gold_set: Dict[Tuple[int,int], int] = {
                (s, e): lbl
                for (s, e), lbl in zip(
                    batch["span_idxs"][b], batch["span_labels"][b])
                if lbl != 13
            }

            # ── Build candidate set (same as predict()) ───────────────────
            cand_mask = (
                torch.tril(torch.ones(sl, sl, device=device, dtype=torch.bool),
                           diagonal=cfg.max_span_len)
                & torch.triu(torch.ones(sl, sl, device=device, dtype=torch.bool))
            )
            combined = s_p.unsqueeze(1) * e_p.unsqueeze(0) * torch.sigmoid(sc)
            combined = combined * cand_mask.float()
            boundary_scores_map: Dict[Tuple[int,int], float] = {}

            flat = combined.view(-1)
            k = min(cfg.top_k_spans, int((flat > 0).sum().item()))

            pred_set: Dict[Tuple[int,int], int] = {}
            pred_conf: Dict[Tuple[int,int], float] = {}

            if k > 0:
                top_vals, top_idx = flat.topk(k)
                cands = []
                for i in range(k):
                    v = top_vals[i].item()
                    s_idx = int(top_idx[i] // sl)
                    e_idx = int(top_idx[i] % sl)
                    boundary_scores_map[(s_idx, e_idx)] = v
                    cands.append((s_idx, e_idx))

                span_logits, _ = model.span_head(hidden[b:b+1], [cands])
                probs = F.softmax(span_logits, dim=-1)   # [N, num_labels]
                pred_labels = span_logits.argmax(-1).tolist()

                for (s, e), lbl, prob_row in zip(cands, pred_labels, probs):
                    if lbl != 13:
                        pred_set[(s, e)] = lbl
                        pred_conf[(s, e)] = prob_row[lbl].item()

            # ── Classify errors ───────────────────────────────────────────
            pred_spans = set(pred_set.keys())
            gold_spans = set(gold_set.keys())

            # False positives: predicted but not gold
            for (s, e) in pred_spans - gold_spans:
                rows.append({
                    "pmid":              pmid,
                    "location":          location,
                    "span_text":         tokenizer.decode(
                                             token_ids[s:e+1],
                                             skip_special_tokens=True),
                    "start_token":       s,
                    "end_token":         e,
                    "error_type":        "FP",
                    "pred_label":        pred_set[(s, e)],
                    "pred_label_name":   _label_name(pred_set[(s, e)]),
                    "gold_label":        None,
                    "gold_label_name":   "—",
                    "boundary_score":    round(boundary_scores_map.get((s,e), 0.0), 4),
                    "span_head_conf":    round(pred_conf.get((s, e), 0.0), 4),
                })

            # False negatives: gold but not predicted
            for (s, e) in gold_spans - pred_spans:
                rows.append({
                    "pmid":              pmid,
                    "location":          location,
                    "span_text":         tokenizer.decode(
                                             token_ids[s:e+1],
                                             skip_special_tokens=True),
                    "start_token":       s,
                    "end_token":         e,
                    "error_type":        "FN",
                    "pred_label":        None,
                    "pred_label_name":   "—",
                    "gold_label":        gold_set[(s, e)],
                    "gold_label_name":   _label_name(gold_set[(s, e)]),
                    "boundary_score":    round(boundary_scores_map.get((s,e), 0.0), 4),
                    "span_head_conf":    None,
                })

            # Wrong label: same span, different label
            for (s, e) in pred_spans & gold_spans:
                p_lbl = pred_set[(s, e)]
                g_lbl = gold_set[(s, e)]
                if p_lbl != g_lbl:
                    rows.append({
                        "pmid":              pmid,
                        "location":          location,
                        "span_text":         tokenizer.decode(
                                                 token_ids[s:e+1],
                                                 skip_special_tokens=True),
                        "start_token":       s,
                        "end_token":         e,
                        "error_type":        "WRONG_LABEL",
                        "pred_label":        p_lbl,
                        "pred_label_name":   _label_name(p_lbl),
                        "gold_label":        g_lbl,
                        "gold_label_name":   _label_name(g_lbl),
                        "boundary_score":    round(boundary_scores_map.get((s,e),0.0), 4),
                        "span_head_conf":    round(pred_conf.get((s, e), 0.0), 4),
                    })

            n_examples += 1

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # ── Summary printout ─────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"Error analysis over {n_examples} examples")
    print(f"{'='*60}")
    counts = df["error_type"].value_counts()
    for etype, cnt in counts.items():
        print(f"  {etype:<14}: {cnt}")

    if "FP" in counts and counts["FP"] > 0:
        print(f"\nTop FP predicted labels (boundary false positives reaching span head):")
        fp_df = df[df["error_type"] == "FP"]
        print(fp_df["pred_label_name"].value_counts().head(10).to_string())
        print(f"\n  Mean boundary_score of FPs: "
              f"{fp_df['boundary_score'].mean():.4f}")
        print(f"  Mean span_head_conf  of FPs: "
              f"{fp_df['span_head_conf'].mean():.4f}")
        print(f"  (High conf + high boundary score → boundary head is overconfident)")
        print(f"  (Low  conf + high boundary score → span head is uncertain, "
              f"consider raising threshold)")

    if "WRONG_LABEL" in counts and counts["WRONG_LABEL"] > 0:
        print(f"\nLabel confusion pairs (gold → pred):")
        wl_df = df[df["error_type"] == "WRONG_LABEL"]
        confusion = (
            wl_df.groupby(["gold_label_name", "pred_label_name"])
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        print(confusion.head(15).to_string(index=False))

    if "FN" in counts and counts["FN"] > 0:
        print(f"\nMost missed gold label types:")
        fn_df = df[df["error_type"] == "FN"]
        print(fn_df["gold_label_name"].value_counts().head(10).to_string())
        n_boundary_miss = (fn_df["boundary_score"] == 0.0).sum()
        print(f"\n  FNs with boundary_score=0 (boundary head missed entirely): "
              f"{n_boundary_miss} / {len(fn_df)}")
        print(f"  FNs with boundary_score>0 (boundary found, span head said None): "
              f"{len(fn_df) - n_boundary_miss} / {len(fn_df)}")

    print(f"{'='*60}\n")
    return df


In [ ]:
# 1. Load the full dictionary
checkpoint = torch.load('/content/stage_3_epoch_9_checkpoint.pth', map_location='cpu')

# 2. Extract the model weights specifically
state_dict = checkpoint['model_state_dict']

# 3. Load those weights into your model
model.load_state_dict(state_dict)
model.eval()
print("Model weights loaded successfully!")

In [ ]:
train_silver_gold_loader = DataLoader(
        train_silver_gold_ds, batch_size=CFG.batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )
model.to('cpu')
model.float()
model.eval()

# 2. Update your CFG to reflect the current device
CFG.device = 'cpu'

# 3. Call without autocast
df = error_analysis(model, train_silver_gold_loader, tokenizer, CFG)
print(df)

In [ ]:
df.to_csv("errors.csv", index=False)

In [ ]:
submission  = format_predictions(model, test_loader, tokenizer, CFG)
submission.to_csv("submission.csv", index=False)

## References

* Sounack, Thomas, et al. "Bioclinical modernbert: A state-of-the-art long-context encoder for biomedical and clinical nlp." arXiv preprint arXiv:2506.10896 (2025).
* Li, Xiaoya, et al. "A unified MRC framework for named entity recognition." Proceedings of the 58th annual meeting of the association for computational linguistics. 2020.
* Gemini, Claude, Chat GPT